<a href="https://colab.research.google.com/github/juanfloresponce2006-dot/Examen01_A-ISI/blob/main/Examen_01A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**KPI 1 CR:** Se escogió un gráfico de tipo tacómetro para una inmediate visualización del rendimiento en porcentaje, un valor alto reduce los costos operativos por re-llamadas y refleja personal altamente capacitado.

**KPI 2 ASA:** Se implementó un gráfico tipo tacómetro para conocer inmediatamente el tiempo en promedio que tardan en contestar los agentes. Es un indicador de que tan eficiente es el TPS.  

**KPI 3 CSAT:** Se utilizó un gráfico tipo tacómetro en una escala del 1 al 10 para una sencilla interpretación. Permite evaluar la satisfacción de la calidad del servicio.

**KPI 4 Performance per Agent:** Fue necesario un gráfico de barras para poner en perspectiva el volumen de llamadas por agente conjunto la tasa de abandono de llamadas del mismo. Permite conocer el rendimiento individual de los agentes respecto a cuantas llamadas hacen.

In [1]:
import pandas as pd
import plotly.express as px

df = pd.read_excel("/content/01 Call-Center-Dataset.xlsx")
df.fillna(0, inplace = True)
df.tail()

,Call Id,Agent,Date,Time,Topic,Answered (Y/N),Resolved,Speed of answer in seconds,AvgTalkDuration,Satisfaction rating
4995,ID4996,Jim,2021-03-31,16:37:55,Payment related,Y,Y,22.0,00:05:40,1.0
4996,ID4997,Diane,2021-03-31,16:45:07,Payment related,Y,Y,100.0,00:03:16,3.0
4997,ID4998,Diane,2021-03-31,16:53:46,Payment related,Y,Y,84.0,00:01:49,4.0
4998,ID4999,Jim,2021-03-31,17:02:24,Streaming,Y,Y,98.0,00:00:58,5.0
4999,ID5000,Diane,2021-03-31,17:39:50,Contract related,N,N,0.0,0,0.0


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Call Id                     5000 non-null   object 
 1   Agent                       5000 non-null   object 
 2   Date                        5000 non-null   object 
 3   Time                        5000 non-null   object 
 4   Topic                       5000 non-null   object 
 5   Answered (Y/N)              5000 non-null   object 
 6   Resolved                    5000 non-null   object 
 7   Speed of answer in seconds  5000 non-null   float64
 8   AvgTalkDuration             5000 non-null   object 
 9   Satisfaction rating         5000 non-null   float64
dtypes: float64(2), object(8)
memory usage: 390.8+ KB


In [3]:
# KPI 4 volumen de llamadas y tasa de abandono*agente
agent_stats = df.groupby('Agent').agg(Total_Llamadas = ('Call Id', 'count'),
                                      Contestadas = ('Answered (Y/N)', lambda x: (x == 'Y').sum()),
                                      No_Contestadas = ('Answered (Y/N)', lambda x: (x == 'N').sum())).reset_index()

agent_stats['Tasa_Abandono_%'] = ((agent_stats['No_Contestadas'] / agent_stats['Total_Llamadas']) * 100).round()
agent_stats


,Agent,Total_Llamadas,Contestadas,No_Contestadas,Tasa_Abandono_%
0,Becky,631,517,114,18.0
1,Dan,633,523,110,17.0
2,Diane,633,501,132,21.0
3,Greg,624,502,122,20.0
4,Jim,666,536,130,20.0
5,Joe,593,484,109,18.0
6,Martha,638,514,124,19.0
7,Stewart,582,477,105,18.0


In [4]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Install kaleido for static image export with plotly
!pip install -U kaleido

df['Speed of answer'] = pd.to_numeric(df['Speed of answer in seconds']) #cambiamos el formato y añanidmos columna pq el dtype son objetos y no numeros
df['Satisfaction rating'] = pd.to_numeric(df['Satisfaction rating']) #lo mismo de arriba


#KPI 1 tasa de resolución en la Llamada (CR)
total_calls = len(df) #devuelve el número total de llamadas (cada fila es una llamada)
resolved_calls = len(df[df['Resolved'] == 'Y']) #devuelve el número total de llamadas que fueron resueltas
cr_rate = (resolved_calls / total_calls) * 100 #cr_rate donde es el porcentaje de llamadas resueltas

#KPI 2 tiempo promedio de respuesta (ASA)
asa_avg = df['Speed of answer'].mean()

#KPI 3 promedio de satisfacción del total de clientes
csat_avg = (df['Satisfaction rating'].mean())*2


#setting el upper dashboard
upperdashboard = make_subplots(rows = 1, cols = 3,
                               specs = [[{'type': 'indicator'}, {'type': 'indicator'}, {'type': 'indicator'}]], #las posiciones para los indicadores
                               subplot_titles = ("","",""))

#call resolution
upperdashboard.add_trace(go.Indicator(mode = "number + gauge",
                                      value = cr_rate,
                                      number = {'suffix': "%"},
                                      gauge = {'axis': {'range': [0, 100]}, 'bar': {'color': "green"}},
                                      title = {'text': "Resolución"}),
                                      row = 1, col = 1)

#avg speed of answer
upperdashboard.add_trace(go.Indicator(mode = "number + gauge",
                                      value = asa_avg,
                                      number = {'suffix': " seg"},
                                      gauge = {'axis': {'range': [0, df['Speed of answer'].max()]}, 'bar': {'color': "blue"}},
                                      title = {'text': "Espera Promedio"}),
                                      row = 1, col = 2)

#client avg satisfaction
upperdashboard.add_trace(go.Indicator(mode = "number + gauge",
                                      value = csat_avg,
                                      gauge = {'axis': {'range': [1, 10]}, 'bar': {'color': "orange"}},
                                      title = {'text': "Satisfacción (1-10)"}),
                                      row = 1, col = 3)


upperdashboard.update_layout(height = 400,
                             title_text = "Panel de Rendimiento Global del Call Center",
                             font=dict(weight = "bold"))
upperdashboard.show()

#la lower part of the dashboard
lowerdashboard = make_subplots(specs = [[{"secondary_y": True}]])


#KPI 4 volumen de llamadas y tasa de abandono*agente
lowerdashboard.add_trace(go.Bar(x = agent_stats['Agent'], y = agent_stats['Total_Llamadas'],
                                name = "Volumen total de Llamadas",
                                marker_color = 'lightblue'),
                                secondary_y = False)

#tasa de abandono
lowerdashboard.add_trace(go.Scatter(x = agent_stats['Agent'], y = agent_stats['Tasa_Abandono_%'],
                                    name = "Tasa de Abandono (%)",
                                    mode = 'lines + markers',
                                    marker_color = 'red'),
                                    secondary_y = True)

lowerdashboard.update_layout(title_text = "Desempeño por Agente: Volumen vs Abandono",
                             xaxis_title = "Agente",
                             font=dict(weight = "bold"))
lowerdashboard.update_yaxes(title_text = "Cantidad de Llamadas", secondary_y = False)
lowerdashboard.show()

/usr/local/lib/python3.12/dist-packages/kaleido/_sync_server.py:11: UserWarning:




This means that static image generation (e.g. `fig.write_image()`) will not work.

Please upgrade Plotly to version 6.1.1 or greater, or downgrade Kaleido to version 0.2.1.

You can however, use the Kaleido API directly which will work with your plotly version. `kaleido.write_fig(...)`, for example. Please see the kaleido documentation.


